# Smart Semsar — furnish the whole apartment with your own furniture (several photos)

ارفعي صور العفش اللي عندك (كل قطعة في صورة)، والشقة كلها تتفرش بيهم في جولة 3D. القطع اللي ملهاش صورة بتيجي من الكتالوج.

**How it works**
1. Every photo is read by the vision model (Gemini, Groq if Gemini is busy): what it is and about how wide.
   You check and fix this in a table.
2. Every photo becomes a 3D model on the diffusion server (Hunyuan3D on Kaggle, `cv_service/furniture_server_kaggle.ipynb`),
   or on TRELLIS.2 if no server address is set.
3. Each piece takes the place of one catalog model of its kind: the first sofa photo replaces the 3-seat sofa,
   a second sofa photo the 2-seat sofa, a second bed the single bed. Every copy of that model in the apartment
   becomes yours (all dining chairs, both double beds ...).
4. The project's own rules place everything, the validator checks it, and the walkthrough is built.

The work is done by `backend/services/multi_furnish.py` (the same code the furniture agent uses);
this notebook adds the page around it.

**Before you run**
- Runtime: CPU is enough (the 3D work happens on the server).
- Secrets (key icon, *Notebook access* on): `HF_TOKEN`, `GEMINI_API_KEY`, `GROQ_API_KEY`, and `SMARTSEMSAR_KEY`
  (the same password the diffusion server uses).
- Start the diffusion server first (Kaggle notebook, **Run All** twice) and put its address in `COLAB_URL` below.

Then **Runtime → Run all**, and open the link the last cell prints.

## 1. Settings

In [ ]:
REPO = "ghadeersalah42/SmartSemsar"
BRANCH = "furniture-merge"
COLAB_URL = ""      # the diffusion server the Kaggle notebook prints; "" = use TRELLIS.2 instead

## 2. Get the code

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:          # secret missing or notebook access off
        return ""

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
os.chdir("/content")
if not os.path.isdir("SmartSemsar"):
    subprocess.run(["git", "clone", "-q", "--depth", "20", "-b", BRANCH, url, "SmartSemsar"], check=True)
os.chdir("/content/SmartSemsar")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Install (about a minute)

In [ ]:
!pip install -q shapely trimesh scipy mapbox_earcut pandas pydantic pillow requests gradio gradio_client

## 4. Keys
Each name the code reads is filled from the first of your secrets that exists. Only *set / NOT SET* is printed.

In [ ]:
# each name the code reads is filled from the first of your secrets that exists
KEYS = {"HF_TOKEN": ["HF_TOKEN", "HF_TOKEN_2", "HF_TOKEN_3"],
        "GEMINI_API_KEY": ["GEMINI_API_KEY", "AGENT_API_KEY_2"],
        "GROQ_API_KEY": ["GROQ_API_KEY", "GROQ_API_KEY_2"],
        "SMARTSEMSAR_COLAB_KEY": ["SMARTSEMSAR_KEY"]}       # the password of the image-to-3D server
for env, names in KEYS.items():
    name = next((n for n in names if secret(n)), None)
    pasted = secret(name) if name else ""
    value = "".join(pasted.split())          # keys never contain spaces or line breaks
    if value:
        os.environ[env] = value
    note = " (had spaces or line breaks from pasting; cleaned)" if value != pasted.strip() else ""
    print(f"{env:22s} {'set from ' + name if value else 'not set'}{note}")
if COLAB_URL:
    os.environ["SMARTSEMSAR_COLAB_URL"] = COLAB_URL

## 5. Which Gemini and Groq models work right now
Google and Groq retire and overload models often. Each model is really called (a short text question,
and a photo for reading furniture); the first one that answers is the one the app uses.

In [ ]:
import sys
sys.path.insert(0, "/content/SmartSemsar")
from backend.services.model_check import pick_working_models
chosen = pick_working_models()

## 6. Is the diffusion server there?

In [ ]:
import sys
sys.path.insert(0, "/content/SmartSemsar")
from backend.services import colab_service
if COLAB_URL:
    try:
        print("server:", colab_service.health())
    except Exception as e:
        print("Server not reachable, so photos will go to TRELLIS.2.", e)
        os.environ.pop("SMARTSEMSAR_COLAB_URL", None)
else:
    print("No COLAB_URL: photos go to TRELLIS.2 (needs HF_TOKEN; daily quota).")

## 7. Many photos → many pieces → the furnished apartment

In [ ]:
# The work is in backend/services/multi_furnish.py; these wrappers fit it to the page below.
import time
from pathlib import Path

from backend.config import setting
from backend.schema.design import DesignPreferences
from backend.schema.staging import load_catalog
from backend.services import vision_service
from backend.services.multi_furnish import Piece, assign_slots, furnish_with_pieces
from backend.services.multi_furnish import raw_model as make_raw_model
from inference import OUT_DIR, ROOT, load_listings

RAW_DIR = OUT_DIR / "_raw_models"      # one raw 3D model per photo, reused when only the table changes


def default_generator():
    return "colab" if setting("SMARTSEMSAR_COLAB_URL") else "trellis"


def read_photo(path, catalog=None):
    """What the vision model sees in one photo -> {"kind", "width_m", "note"} (kind "" if unknown)."""
    if not vision_service.is_configured():
        return {"kind": "", "width_m": None, "note": "no vision key: fill in 'what is it' yourself"}
    try:
        seen = vision_service.analyze_photo(path, catalog)
    except vision_service.VisionError as e:
        return {"kind": "", "width_m": None, "note": f"could not read it ({str(e)[:70]}): fill it in yourself"}
    note = seen.name or seen.scene
    if not seen.usable:
        note += f" - {seen.reason}"
    return {"kind": seen.kind or "", "width_m": seen.width_m, "note": note}


def raw_model(photo, generator, say=print):
    return make_raw_model(photo, generator, RAW_DIR, say)


def place_pieces(property_id, pieces, generator, prefs=None, planner="rules", say=print):
    """pieces: [{label, slot, raw, width_m, turn}] -> (walkthrough path or None, staging)."""
    rows = load_listings()
    listing = rows[rows["property_id"] == property_id].iloc[0].to_dict()
    say(f"Plan: {listing['title'].strip()}")
    result = furnish_with_pieces(ROOT / listing["plan_path"],
                                 [Piece(str(p["raw"]), p["slot"], p.get("width_m"), p.get("turn", 0), p["label"])
                                  for p in pieces],
                                 OUT_DIR / property_id / "multi", prefs, planner, generator, listing, say=say)
    for p in pieces:
        item = result.mine.get(p["label"])
        if item:
            p["fitted"] = item.mesh            # the piece as it stands in the apartment (front = +z)
    return result.walkthrough, result.staging

## 8. The app
1. Upload all your photos → **1. Read the photos**.
2. Check the table: *what is it* must be one of the catalog kinds (the message lists them), *width* in meters,
   *turn* 0/90/180/270 if a piece faces the wrong way. Leave *what is it* empty to skip a photo.
3. **2. Furnish the apartment**. Each coloured piece takes about 2–3 minutes on the server; a second run with
   the same photos reuses the 3D models, so changing a width or the apartment is quick.
   Above it: *How much furniture*, *Must have*, *Leave out*, *Dining seats*, a *Style description*, and
   *Who arranges the furniture* (the rules, or AI: Gemini, Groq if Gemini is busy; the rules take over if both fail).
4. **Turn it (if it faces sideways)**: pick a piece (the list fills when the photos are read) and choose 0/90/180/270. After Furnish you can also look at it in 3D (drag to rotate; the side facing you is its
   front), choose *Turn it* 0/90/180/270 if it faces the wall or sideways, and press Furnish again.

In [ ]:
# The app: upload several photos -> check the table -> furnish the whole apartment with them
import math

import gradio as gr
import pandas as pd

from backend.services.llm_furnisher import detect_llm

OUT_DIR.mkdir(exist_ok=True)
gr.set_static_paths(paths=[OUT_DIR])          # lets the page load the walkthrough it builds

LISTINGS = load_listings()
LISTING_CHOICES = [(f"{r.property_id} · {str(r.title).strip()[:55]} · {r.pf_area_sqm:.0f} m² · {r.pf_bedrooms} bed",
                    r.property_id) for r in LISTINGS.itertuples()]
KINDS = sorted({i.kind for i in load_catalog().items})
COLUMNS = ["photo", "what is it", "width (m)", "turn", "note"]
DEFAULT_TURN = 180     # every piece starts turned 180 degrees; change it per piece below
GENERATOR_CHOICES = [("Diffusion server (Hunyuan3D on Kaggle/Colab, coloured)", "colab"),
                     ("TRELLIS.2 on Hugging Face (daily quota)", "trellis")]
AUTO = "Auto"
PLANNERS = [("Rules", "rules"), ("AI designs it (Gemini, Groq if Gemini is busy)", "llm")]
PLANNER_MODEL = detect_llm()
EMPTY_VIEW = "<div style='padding:40px;text-align:center;color:#888'>The furnished apartment appears here.</div>"


def _path(f):
    return f if isinstance(f, str) else getattr(f, "name", None) or f["path"]


def _number(value, default=None):
    try:
        value = float(value)
        return default if math.isnan(value) else value
    except (TypeError, ValueError):
        return default


def read_photos(files):
    if not files:
        return (pd.DataFrame(columns=COLUMNS), "Upload your furniture photos first (one piece per photo).", [],
                gr.update(choices=[], value=None))
    catalog, rows, paths = load_catalog(), [], [_path(f) for f in files]
    for path in paths:
        seen = read_photo(path, catalog)
        rows.append([Path(path).name, seen["kind"], seen["width_m"], DEFAULT_TURN, seen["note"]])
    missing = sum(not r[1] for r in rows)
    msg = f"Read {len(rows)} photo(s). Check 'what is it' (one of: {', '.join(KINDS)}) and the width, then Furnish."
    if missing:
        msg += f"\n{missing} photo(s) still need 'what is it'."
    names = [Path(p).name for p in paths]
    return pd.DataFrame(rows, columns=COLUMNS), msg, paths, gr.update(choices=names, value=names[0] if names else None)


def furnish(paths, table, property_id, density, must_have, exclude, dining_seats, style_brief, planner, generator):
    """Streams (report, view, download, pieces, piece choices) while it works."""
    lines = []
    say = lines.append
    keep = gr.update()

    def out(view=keep, download=None, pieces=keep, choices=keep):
        return "\n".join(lines), view, download, pieces, choices

    if not paths:
        say("Upload photos and press 'Read the photos' first.")
        yield out(EMPTY_VIEW)
        return
    table = table if isinstance(table, pd.DataFrame) else pd.DataFrame(table, columns=COLUMNS)
    pieces = []
    for path, (_, row) in zip(paths, table.iterrows()):
        turn = int(_number(row["turn"], DEFAULT_TURN)) % 360
        pieces.append({"label": Path(path).name, "photo": path, "kind": str(row["what is it"] or "").strip(),
                       "width_m": _number(row["width (m)"]), "turn": turn if turn in (0, 90, 180, 270) else DEFAULT_TURN})
    prefs = DesignPreferences.from_loose({
        "density": density, "must_have": must_have or [], "exclude": exclude or [],
        "dining_seats": None if dining_seats == AUTO else int(dining_seats),
        "style_brief": (style_brief or "").strip() or None})
    say(f"{len(pieces)} photo(s), 3D with {generator}, arranged by {planner}")
    pieces = assign_slots(pieces, load_catalog(), say)
    if not pieces:
        say("Nothing to place.")
        yield out(EMPTY_VIEW)
        return
    ready = []
    for n, p in enumerate(pieces, 1):
        say(f"{n}/{len(pieces)} {p['label']} ({p['slot']}): making the 3D model...")
        yield out()
        try:
            p["raw"] = raw_model(p["photo"], generator, say)
            ready.append(p)
        except Exception as e:          # server down, quota used up, bad photo ...
            say(f"     failed: {type(e).__name__}: {str(e)[:160]}")
        yield out()
    if not ready:
        say("No 3D model could be made.")
        yield out(EMPTY_VIEW)
        return
    say("Placing everything in the apartment...")
    yield out()
    try:
        html, _ = place_pieces(property_id, ready, generator, prefs, planner, say)
    except Exception as e:
        say(f"error: {type(e).__name__}: {e}")
        yield out(EMPTY_VIEW)
        return
    made = [{"label": p["label"], "slot": p["slot"], "turn": p["turn"], "fitted": p["fitted"]}
            for p in ready if p.get("fitted")]
    choices = gr.update(choices=[p["label"] for p in made], value=made[0]["label"] if made else None)
    if html is None:
        yield out(EMPTY_VIEW, None, made, choices)
        return
    src = f"/gradio_api/file={Path(html).resolve().as_posix()}?t={int(time.time())}"
    frame = (f'<iframe src="{src}" title="3D walkthrough" allow="fullscreen" allowfullscreen '
             f'style="width:100%;height:680px;border:0;border-radius:8px"></iframe>')
    say("\nCheck each piece below: if one faces the wall or sideways, pick its turn and press Furnish again "
        "(the 3D models are reused, so it is quick).")
    yield out(frame, str(html), made, choices)


def show_piece(label, made, table, paths):
    """The chosen piece: its turn from the table, and its 3D model once Furnish has made it
    (as it stands in the apartment: the side facing you at the start is its front)."""
    table = table if isinstance(table, pd.DataFrame) else pd.DataFrame(table, columns=COLUMNS)
    row = next((i for i, p in enumerate(paths or []) if Path(p).name == label), None)
    turn = int(_number(table.iloc[row]["turn"], DEFAULT_TURN)) if row is not None and row < len(table) else DEFAULT_TURN
    turn = str(turn if turn in (0, 90, 180, 270) else DEFAULT_TURN)
    piece = next((p for p in made or [] if p["label"] == label), None)
    if piece is None:
        return None, turn, f"**{label}**: its 3D model appears here after Furnish."
    return (piece["fitted"], turn,
            f"**{label}** is your `{piece['slot']}`. Drag to look around; the side facing you at the start is its front.")


def set_turn(label, turn, table, paths):
    """Write the chosen turn into the table row of that photo."""
    table = table if isinstance(table, pd.DataFrame) else pd.DataFrame(table, columns=COLUMNS)
    table = table.copy()
    for i, path in enumerate(paths or []):
        if Path(path).name == label and i < len(table):
            table.iloc[i, COLUMNS.index("turn")] = int(turn)
    return table


with gr.Blocks(title="Smart Semsar - furnish with your own furniture") as demo:
    gr.Markdown("# Smart Semsar — furnish the whole apartment with your own furniture\n"
                "Upload a photo of each piece you have (one piece per photo), check the table, and walk through "
                "the apartment furnished with them. Pieces you have no photo of come from the catalog.")
    paths = gr.State([])
    made = gr.State([])
    with gr.Row():
        with gr.Column(scale=1, min_width=360):
            listing = gr.Dropdown(LISTING_CHOICES, value="PROP_1002", label="Apartment")
            photos = gr.File(file_count="multiple", file_types=["image"], label="Your furniture photos")
            read_btn = gr.Button("1. Read the photos")
            table = gr.Dataframe(headers=COLUMNS, datatype=["str", "str", "number", "number", "str"],
                                 col_count=(len(COLUMNS), "fixed"), interactive=True, wrap=True,
                                 label="Your pieces (edit 'what is it', width in meters, turn 0/90/180/270)")
            density = gr.Radio(["minimal", "normal", "full"], value="normal", label="How much furniture")
            must_have = gr.CheckboxGroup(KINDS, label="Must have")
            exclude = gr.CheckboxGroup(KINDS, label="Leave out")
            dining_seats = gr.Radio([AUTO, "4", "6"], value=AUTO, label="Dining seats")
            style_brief = gr.Textbox(label="Style description (used by the Gemini planner)",
                                     placeholder="e.g. calm modern, light wood, grey fabric, a reading corner")
            planner = gr.Radio(PLANNERS, value="rules", label="Who arranges the furniture")
            generator = gr.Radio(GENERATOR_CHOICES, value=default_generator(), label="Photo to 3D with")
            go = gr.Button("2. Furnish the apartment", variant="primary")
            with gr.Group():
                piece = gr.Dropdown([], label="Piece to turn", info="Filled when the photos are read")
                turn = gr.Radio(["0", "90", "180", "270"], value=str(DEFAULT_TURN), label="Turn it (if it faces sideways)")
                piece_note = gr.Markdown()
                piece_view = gr.Model3D(label="Your piece (drag to rotate)", height=300,
                                        clear_color=(0.15, 0.15, 0.15, 1.0))
        with gr.Column(scale=3):
            view = gr.HTML(EMPTY_VIEW)
            report = gr.Textbox(label="What happened", lines=14, max_lines=30)
            download = gr.File(label="Walkthrough page (opens offline in any browser)")
    gr.Markdown(f"Keys found: GEMINI_API_KEY {'yes' if setting('GEMINI_API_KEY') else 'no'} · "
                f"GROQ_API_KEY {'yes' if setting('GROQ_API_KEY') else 'no'} · "
                f"planner model: {PLANNER_MODEL.name if PLANNER_MODEL else 'none (rules only)'}")
    read_btn.click(read_photos, [photos], [table, report, paths, piece])
    go.click(furnish, [paths, table, listing, density, must_have, exclude, dining_seats, style_brief, planner, generator], [report, view, download, made, piece])
    piece.change(show_piece, [piece, made, table, paths], [piece_view, turn, piece_note])
    turn.input(set_turn, [piece, turn, table, paths], [table])

demo.queue().launch(share=True, height=1200)